 Train + test SDXL LoRA
Train an SDXL LoRA from a Drive dataset, then compare epochs.

1. **Runtime > Change runtime type > T4 GPU**
2. **Secrets** (key icon, left), allow notebook access: `HF_TOKEN` (gated or private repos),
   `CIVITAI_TOKEN` (Civitai downloads).
3. Run **Setup** (re-runs skip finished steps), then the cells below in order.

In [ ]:
#@title Setup: repo, venv, secrets, Drive
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "step5-notebooks"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/SDXLR"  #@param {type:"string"}
GROUPS = 'caption,train-sdxl'  # venv install groups for this notebook (None = no shared venv)
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # follow BRANCH even if an earlier run cloned another one
    subprocess.run(["git", "fetch", "-q", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
if GROUPS is not None:
    r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap", "--groups", GROUPS], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONPATH": str(REPO / "src")})
    print(r.stdout[-3000:])
    assert r.returncode == 0, "setup failed (see above)"
found = nb.load_secrets()
print("secrets:", ", ".join(f"{k} {'found' if v else 'missing'}" for k, v in found.items() if k in ('HF_TOKEN', 'CIVITAI_TOKEN')))
ROOT = nb.drive_mount(DRIVE_ROOT) if USE_DRIVE else None
print("drive:", ROOT or "off")

# 1. Dataset
Copies `SDXLR/datasets/<DS_NAME>` from Drive to the VM (training reads many small files; Drive is slow for that).
Photos without a caption can be captioned here (Qwen3-VL 4B, identity style); the captioning notebook has more options.

In [ ]:
#@title Dataset
DS_NAME = "my_dataset"  #@param {type:"string"}
CAPTION_MISSING = True  #@param {type:"boolean"}
TRIGGER = "ohwx"  #@param {type:"string"}
CLASS_WORD = "woman"  #@param {type:"string"}
from sdxlr import drive, plan, paths
IMGS = (".png", ".jpg", ".jpeg", ".webp")
LOCAL = paths.DATASETS / DS_NAME
LOCAL.mkdir(parents=True, exist_ok=True)
DRIVE_DS = ROOT / "datasets" / DS_NAME if ROOT else None
if DRIVE_DS:
    assert DRIVE_DS.exists(), f"{DRIVE_DS} not found: run the captioning notebook first, or check DS_NAME"
    print(f"from Drive: {drive.pull(DRIVE_DS, LOCAL)} files copied")
imgs = sorted(p for p in LOCAL.iterdir() if p.suffix.lower() in IMGS)
missing = [p for p in imgs if not p.with_suffix(".txt").exists()]
print(f"{len(imgs)} images, {len(imgs) - len(missing)} captioned")
if missing and CAPTION_MISSING:
    args = ["-m", "sdxlr.caption", "--images", LOCAL, "--style", "id", "--target", "sdxl",
            "--trigger", TRIGGER, "--class-word", CLASS_WORD]
    nb.run(args + (["--mirror", DRIVE_DS] if DRIVE_DS else []))

# 2. Settings
**PRESET**: `best` = the measured settings (1024 px, about 30 min for any photo count);
`fast` = 768 px, text encoders frozen, 10 epochs (an estimate until measured).
Leave EPOCHS / REPEATS / DIM / LR at 0 for the preset values; repeats 0 = auto from your photo count.

In [ ]:
#@title Settings
BASE = "sdxl-base"  #@param ["sdxl-base", "pony-v6", "illustrious-0.1", "noobai-eps-1.1", "noobai-vpred-1.0"]
CUSTOM_BASE = ""  #@param {type:"string"}
#@markdown CUSTOM_BASE: an HF file `org/repo/file.safetensors`, a Civitai or https link, or a Drive path; overrides BASE.
SUBJECT = "person"  #@param ["person", "character/object", "style", "concept/pose"]
PRESET = "best"  #@param ["best", "fast"]
RUN = "my_lora"  #@param {type:"string"}
EPOCHS = 0  #@param {type:"integer"}
REPEATS = 0  #@param {type:"integer"}
DIM = 0  #@param {type:"integer"}
LR = 0.0  #@param {type:"number"}
RESUME = True  #@param {type:"boolean"}
#@markdown RESUME continues a stopped run with the same RUN name from its state on Drive.
import yaml
cfg = yaml.safe_load((REPO / "configs/train" / ("sdxl_person.yaml" if PRESET == "best" else "sdxl_fast.yaml")).read_text())
BASE_SRC = CUSTOM_BASE or BASE
sets = []
if EPOCHS: sets.append(f"epochs={EPOCHS}")
sets.append(f"repeats={REPEATS}")  # 0 = auto
if DIM: sets += [f"network_dim={DIM}", f"network_alpha={round(DIM * cfg['network_alpha'] / cfg['network_dim'])}"]  # keeps alpha/dim
if LR: sets.append(f"unet_lr={LR}")
n = len(sorted(p for p in LOCAL.iterdir() if p.suffix.lower() in IMGS))
p = plan.train("sdxl", SUBJECT, n, cfg["resolution"], cfg["batch_size"], EPOCHS or cfg["epochs"], REPEATS)
print(p.summary())
print(f"LoRA scale = alpha/dim x strength = {plan.lora_scale(cfg['network_alpha'], cfg['network_dim']):.2f} x strength")

# 3. Train
Every epoch is copied to `SDXLR/loras/sdxl/<RUN>/` and the newest state to `SDXLR/state/<RUN>/` as soon as it is saved.
If Colab disconnects, run Setup, Dataset, Settings and this cell again: it continues where it stopped.

In [ ]:
#@title Train
args = ["-m", "sdxlr.train", "--name", RUN, "--images", LOCAL, "--preset", PRESET, "--model", BASE_SRC,
        "--subject", SUBJECT, "--trigger", TRIGGER, "--class-word", CLASS_WORD, "--set", *sets]
if ROOT:
    args += ["--drive", ROOT] + (["--resume"] if RESUME else [])
nb.run(args)

# 4. Test
A grid: rows = the last epochs (x prompts), columns = LoRA strengths, same seed everywhere.
Time is about 20 s per image (SDXL, 20 steps) plus the model load.

In [ ]:
#@title Test
EPOCHS_TO_TEST = "last:3"  #@param {type:"string"}
#@markdown last:K, all, or a list like 6,10,15
STRENGTHS = "0.6 0.8 1.0"  #@param {type:"string"}
PROMPT_1 = "photo of {t} {c}, portrait, natural light, sharp focus"  #@param {type:"string"}
PROMPT_2 = "photo of {t} {c} walking in a city street, full body"  #@param {type:"string"}
NEGATIVE = "blurry, lowres, deformed"  #@param {type:"string"}
STEPS = 20  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
prompts = [x.replace("{t}", TRIGGER).replace("{c}", CLASS_WORD) for x in (PROMPT_1, PROMPT_2) if x.strip()]
run_dir = paths.WORK / "runs" / RUN
if ROOT and not any(run_dir.glob("*.safetensors")):  # a new VM: test what Drive has
    drive.pull(ROOT / "loras" / "sdxl" / RUN, run_dir)
args = ["-m", "sdxlr.compare", "--name", RUN, "--model", paths.SDXL_BASES.get(BASE_SRC, BASE_SRC),
        "--lora-dir", run_dir, "--epochs", EPOCHS_TO_TEST, "--strengths", *STRENGTHS.split(),
        "--prompts", *prompts, "--negative", NEGATIVE, "--steps", STEPS, "--seed", SEED]
if ROOT:
    args += ["--drive-out", ROOT / "compare" / RUN]
nb.run(args)
from IPython.display import Image as _Img, display
display(_Img(filename=str(paths.WORK / "compare" / RUN / "grid.png")))

# 5. Pick
Copies the epoch you like to `SDXLR/loras/sdxl/<RUN>_e<N>.safetensors` (with its trigger-word sidecar).
Fooocus and the Z-Image app list that folder.

In [ ]:
#@title Pick an epoch
PICK_EPOCH = 0  #@param {type:"integer"}
from sdxlr import loraformat
dst = loraformat.pick(run_dir, PICK_EPOCH, (ROOT or paths.WORK) / "loras" / "sdxl", RUN)
print(f"saved {dst}")

# Finish
Drive writes back lazily: run this before closing the tab, then **Runtime > Disconnect and delete runtime**.

In [ ]:
#@title Save to Drive and finish
if ROOT:
    drive.flush_and_unmount()